In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from openai import OpenAI
import warnings
import re
import pathlib
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)

STORE_PATH = "./data/train_store.pkl"

pd.set_option("display.max_colwidth", 100)


warnings.filterwarnings("ignore")

client = OpenAI()

https://aclanthology.org/W19-3003.pdf

In [11]:
# Send the post and instructions to OpenAI API

def create_prompt(post):

    return f"""
POST: {post}

Step 1: First carefully analyze the POST.

Step 2: Then, carefully check the thoughts pattern, feelings, logistics, and context using the following guideline:

Thoughts includes not only explicit ideation but also, e.g., feeling they are a burden to others or having a “f*** it” (screw it, game over, farewell) thought pattern.
Feelings includes, e.g., a lack of hope for things to get better, or a sense of agitation or impulsivity (mixed depressive state, Popovic et al. (2015)).
Logistics includes, e.g., talking about methods of attempting suicide (even if not planning), or having access to lethal means like firearms.
Context includes, e.g. previous attempts, a significant life change, or isolation from friends and family.

Step 3: Using the knowledge of Step 1 and Step 2, decide a LABEL for the POST based on the following criteria:

No Risk: I don’t see evidence that this person is at risk for suicide. 
Low Risk: There may be some factors here that could suggest risk, but I don’t really think this person is at much of a risk of suicide.
Moderate Risk: I see indications that there could be a genuine risk of this person making a suicide attempt.
Severe Risk: I believe this person is at high risk of attempting suicide in the near future. 
Unknown: It is impossible to classify this into one of the risk categories. 

Step 4: Output your reasoning and your LABEL classification on new lines. Keep your reasoning concise. Prefix your reasoning with the key "Reasoning:". Prefix your LABEL with the key "Label:".
"""

In [ ]:
df = pd.read_csv("materials/annotation_data/train_annotate.csv")

if pathlib.Path(STORE_PATH).exists():
    
    df_store = pd.read_pickle("./data/train_store.pkl")

    df = df.loc[~df["id"].isin(df_store["id"])]

else:
    df_store = None
    
df_remaining = df.copy()

In [ ]:
df = df_remaining.copy()

if df.shape[0] != 0:
    
    df = df.iloc[0:20]

    records = []

    for record in df.to_dict(orient="records"):

        logging.info(f"Processing {record['id']}")
        
        text = record["text"]

        prompt = create_prompt(text)

        response = client.responses.create(
            model="gpt-5.4",
            input=prompt,
            temperature=0,
        )

        reasoning = re.search(r"Reasoning:(.+)", response.output_text).group(1)

        label = re.search(r"Label:(.+)", response.output_text).group(1)


        record["reasoning"] = reasoning

        record["label"] = label

        records.append(record)

    df_result = pd.DataFrame(records)

    df = pd.concat([df_store, df_result])

    df.to_pickle(STORE_PATH)

In [ ]:
df = pd.read_pickle(STORE_PATH)

assert not df['id'].duplicated().any()

print(df.shape[0])

df['label'].value_counts()


In [ ]:
df['label'].hist()

In [ ]:
df = pd.read_pickle(STORE_PATH)

df.to_csv("resample_test_labels.csv", index=False)